# Banking Customer Churn Analysis & Retention Insights

## 1. Introduction and Problem Statement
A bank loses revenue when customers close their accounts or stop using its services. This is called **customer churn**. This project analyses customer records to identify patterns associated with customers leaving the bank.

**Objective:** perform data preprocessing, exploratory data analysis (EDA) and visualization on a banking customer dataset. No prediction model is built.

**Analytical questions**
1. What is the overall churn rate?
2. Are older customers more likely to leave?
3. Do inactive customers leave more often than active ones?
4. Does geography (country) relate to churn?
5. Does the number of products a customer holds relate to churn?
6. Does gender relate to churn?

## 2. Dataset Selection and Description
### 2.1 Dataset Source

The dataset selected for this project is the Banking Customer Churn Modelling dataset, obtained from Kaggle.

**Source URL:** https://www.kaggle.com/datasets/shantanudhakadd/bank-customer-churn-prediction

### 2.2 Dataset Description

The dataset contains information about 10,000 bank customers and includes 14 attributes related to customer demographics, financial information, banking relationships, and customer status.

The target attribute is `Exited`, which indicates whether a customer has left the bank.

- 0 represents customers who stayed.
- 1 represents customers who exited.

### 2.3 Purpose of Dataset Selection

This dataset was selected because it provides a suitable combination of numerical and categorical attributes for investigating customer churn patterns. It allows us to explore customer characteristics, identify differences between retained and exited customers, and derive useful business insights.

### 2.4 Analytical Objectives

The analysis focuses on understanding customer behaviour, examining factors associated with churn, and identifying patterns that may help banking institutions improve customer retention.


## 3. Data preprocessing

importing libraries

In [52]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File handling
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Visualization settings
sns.set_theme(style="whitegrid")

print("All libraries imported successfully!")

All libraries imported successfully!


loading dataset and examining structure

In [54]:
# Locate the dataset
file_candidates = [
    Path("../dataset/Churn_Modelling.csv"),
    Path("dataset/Churn_Modelling.csv")
]

file_path = next(
    (path for path in file_candidates if path.exists()),
    None
)

if file_path is None:
    raise FileNotFoundError("Dataset file not found. Check its location.")

df = pd.read_csv("../dataset/Churn_Modelling.csv")

print("Dimensions (rows, columns):", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nFirst five records:")
display(df.head())
df.info()


Dimensions (rows, columns): (10000, 14)

Column names: ['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']

Data types:
RowNumber            int64
CustomerId           int64
Surname                str
CreditScore          int64
Geography              str
Gender                 str
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object

First five records:


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.1 MB


### 1. missing values

In [56]:
missing_summary = pd.DataFrame({"Missing count": df.isnull().sum(),
                                "Missing %": (df.isnull().mean() * 100).round(2)})
display(missing_summary)
print("Total missing values:", df.isnull().sum().sum())

,Missing count,Missing %
RowNumber,0,0.0
CustomerId,0,0.0
Surname,0,0.0
CreditScore,0,0.0
Geography,0,0.0
Gender,0,0.0
Age,0,0.0
Tenure,0,0.0
Balance,0,0.0
NumOfProducts,0,0.0


Total missing values: 0


### 2. duplicate records

In [57]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate CustomerIds:", df["CustomerId"].duplicated().sum())

Duplicate rows: 0
Duplicate CustomerIds: 0


### dropping useless colums

In [ ]:
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])

### 3. checking datatype 


In [62]:
for col in ["Exited", "HasCrCard", "IsActiveMember"]:
    df[col] = df[col].astype(int)

# Readable label columns for charts (numeric flags are kept for calculations)
df["Status"] = df["Exited"].map({0: "Retained", 1: "Churned"})
df["ActiveLabel"] = df["IsActiveMember"].map({0: "Inactive", 1: "Active"})
print(df.dtypes)

CreditScore          int64
Geography              str
Gender                 str
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
Status                 str
ActiveLabel            str
dtype: object


### correcting inconsistencies

In [65]:
for col in ["Geography", "Gender"]:
    df[col] = df[col].str.strip().str.title()
    print(df[col].value_counts(), "\n")

print("Age range:", df["Age"].min(), "-", df["Age"].max())
print("Credit score range:", df["CreditScore"].min(), "-", df["CreditScore"].max())
print("Negative balances:", (df["Balance"] < 0).sum())
print("Tenure range:", df["Tenure"].min(), "-", df["Tenure"].max())
print("NumOfProducts values:", sorted(int(x) for x in df["NumOfProducts"].unique()))

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64 

Gender
Male      5457
Female    4543
Name: count, dtype: int64 

Age range: 18 - 92
Credit score range: 350 - 850
Negative balances: 0
Tenure range: 0 - 10
NumOfProducts values: [1, 2, 3, 4]


### 4. Outliers (IQR method)

In [66]:
out_cols = ["CreditScore", "Age", "Balance", "EstimatedSalary", "Tenure"]
rows = []
for col in out_cols:
    q1, q3 = df[col].quantile([.25, .75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((df[col] < lo) | (df[col] > hi)).sum()
    rows.append([col, round(lo, 1), round(hi, 1), n, round(n / len(df) * 100, 2)])
outlier_summary = pd.DataFrame(rows, columns=["Column", "Lower bound", "Upper bound", "Outliers", "% of data"])
display(outlier_summary)

,Column,Lower bound,Upper bound,Outliers,% of data
0,CreditScore,383.0,919.0,15,0.15
1,Age,14.0,62.0,359,3.59
2,Balance,-191466.4,319110.6,0,0.00
3,EstimatedSalary,-96577.1,296967.5,0,0.00
4,Tenure,-3.0,13.0,0,0.00


###  Final cleaned dataset

In [ ]:
print("Final shape:", df.shape)
df.info()

display(df.head())

Final shape: (10000, 13)
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  str    
 2   Gender           10000 non-null  str    
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
 11  Status           10000 non-null  str    
 12  ActiveLabel      10000 non-null  str    
dtypes: float64(2), int64(7), str(4)
memory usage: 1015.8 KB


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Status,ActiveLabel
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,Churned,Active
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,Retained,Active
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,Churned,Inactive
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,Retained,Inactive
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,Retained,Active


### Preprocessing summary and decisions

| Check | Result | Decision |
|---|---|---|
| Missing values | 0 in all columns (0.00%) | No imputation needed |
| Duplicate rows / CustomerIds | 0 / 0 | No records removed |
| Identifier columns | `RowNumber`, `CustomerId`, `Surname` have no analytical meaning for churn patterns | Dropped |
| Data types | Numeric columns are `int64`/`float64`; `Geography` and `Gender` are text; binary flags are 0/1 integers | Kept flags numeric for calculations and correlation; added `Status` and `ActiveLabel` for readable charts |
| Inconsistencies | Geography contains only France, Germany, Spain; Gender only Male, Female; no spelling or casing issues | No correction needed |
| Logical ranges | Age 18-92, CreditScore 350-850, Tenure 0-10, NumOfProducts 1-4, no negative balances | All values valid |

**Outliers (IQR method):**

| Column | Lower bound | Upper bound | Outliers | % of data |
|---|---|---|---|---|
| CreditScore | 383.0 | 919.0 | 15 | 0.15% |
| Age | 14.0 | 62.0 | 359 | 3.59% |
| Balance | -191,466.4 | 319,110.6 | 0 | 0.00% |
| EstimatedSalary | -96,577.1 | 296,967.5 | 0 | 0.00% |
| Tenure | -3.0 | 13.0 | 0 | 0.00% |

Outliers appear only in `Age` (customers older than 62) and `CreditScore` (a few very low scores). **All were retained** because they are genuine customers rather than data errors, age is a core analytical question so removing older customers would hide a real pattern, and the CreditScore outliers are only 0.15% of the data. Many customers have a balance of exactly 0; this is a genuine feature (empty accounts), not a missing value.

## 3.Exploratory Data Analysis (EDA)

### 3.1 Descriptive Statistics

In [68]:
num_all = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]
display(df[num_all].describe().round(2))

,CreditScore,Age,Tenure,Balance,NumOfProducts,EstimatedSalary
count,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00
mean,650.53,38.92,5.01,76485.89,1.53,100090.24
std,96.65,10.49,2.89,62397.41,0.58,57510.49
min,350.00,18.00,0.00,0.00,1.00,11.58
25%,584.00,32.00,3.00,0.00,1.00,51002.11
50%,652.00,37.00,5.00,97198.54,1.00,100193.92
75%,718.00,44.00,7.00,127644.24,2.00,149388.25
max,850.00,92.00,10.00,250898.09,4.00,199992.48


**summary**
- **CreditScore:** mean (650.5) and median (652) are almost equal, so scores are roughly symmetric. The middle 50% of customers score between 584 and 718.
- **Age:** mean (38.9) is above the median (37), a right skew from a smaller group of older customers (up to 92). Most customers are between 32 and 44.
- **Tenure:** mean and median are both about 5 years; customers are spread fairly evenly across 0-10 years.
- **Balance:** the mean (76,486) is well below the median (97,199) and the 25th percentile is 0, so at least a quarter of customers hold an empty account.
- **NumOfProducts:** the median is 1 and 75% of customers hold 2 or fewer products.
- **EstimatedSalary:** mean and median are both about 100,000, with a wide even spread from about 12 to 200,000.

### 3.2 Numerical Attribute Analysis

In [69]:
num_cols = ["Age", "CreditScore", "Balance", "Tenure", "EstimatedSalary"]
stats = df[num_cols].agg(["mean", "median", "std", "min", "max", "skew"]).T.round(2)
stats["Q1"] = df[num_cols].quantile(0.25).round(2)
stats["Q3"] = df[num_cols].quantile(0.75).round(2)
display(stats)

zero_bal = (df["Balance"] == 0).sum()
print(f"Customers with zero balance: {zero_bal} ({zero_bal / len(df) * 100:.2f}%)")
print("Mean balance (non-zero only):", round(df.loc[df["Balance"] > 0, "Balance"].mean(), 2))

,mean,median,std,min,max,skew,Q1,Q3
Age,38.92,37.00,10.49,18.00,92.00,1.01,32.00,44.00
CreditScore,650.53,652.00,96.65,350.00,850.00,-0.07,584.00,718.00
Balance,76485.89,97198.54,62397.41,0.00,250898.09,-0.14,0.00,127644.24
Tenure,5.01,5.00,2.89,0.00,10.00,0.01,3.00,7.00
EstimatedSalary,100090.24,100193.92,57510.49,11.58,199992.48,0.00,51002.11,149388.25


Customers with zero balance: 3617 (36.17%)
Mean balance (non-zero only): 119827.49


### 3.3 categorical attribute analysis

In [70]:
for col in ["Geography", "Gender", "Exited"]:
    t = pd.DataFrame({"Count": df[col].value_counts(),
                      "Proportion %": (df[col].value_counts(normalize=True) * 100).round(2)})
    print(f"--- {col} ---")
    display(t)

--- Geography ---


,Count,Proportion %
Geography,,
France,5014,50.14
Germany,2509,25.09
Spain,2477,24.77


--- Gender ---


,Count,Proportion %
Gender,,
Male,5457,54.57
Female,4543,45.43


--- Exited ---


,Count,Proportion %
Exited,,
0,7963,79.63
1,2037,20.37


### 3.4 Investigation of relationships and correlations:

In [73]:
df["AgeGroup"] = pd.cut(df["Age"], bins=[17, 30, 40, 50, 60, 100],
                        labels=["18-30", "31-40", "41-50", "51-60", "60+"])

for col in ["Geography", "Gender", "IsActiveMember", "NumOfProducts", "AgeGroup"]:
    t = df.groupby(col, observed=True)["Exited"].agg(Customers="count", ChurnRate="mean")
    t["ChurnRate"] = (t["ChurnRate"] * 100).round(2)
    print(f"Churn rate by {col} (overall = {df['Exited'].mean() * 100:.2f}%)")
    display(t)

display(df.groupby("Status")[["Age", "CreditScore", "Balance", "Tenure", "EstimatedSalary"]].mean().round(2))

Churn rate by Geography (overall = 20.37%)


,Customers,ChurnRate
Geography,,
France,5014,16.15
Germany,2509,32.44
Spain,2477,16.67


Churn rate by Gender (overall = 20.37%)


,Customers,ChurnRate
Gender,,
Female,4543,25.07
Male,5457,16.46


Churn rate by IsActiveMember (overall = 20.37%)


,Customers,ChurnRate
IsActiveMember,,
0,4849,26.85
1,5151,14.27


Churn rate by NumOfProducts (overall = 20.37%)


,Customers,ChurnRate
NumOfProducts,,
1,5084,27.71
2,4590,7.58
3,266,82.71
4,60,100.00


Churn rate by AgeGroup (overall = 20.37%)


,Customers,ChurnRate
AgeGroup,,
18-30,1968,7.52
31-40,4451,12.09
41-50,2320,33.97
51-60,797,56.21
60+,464,24.78


,Age,CreditScore,Balance,Tenure,EstimatedSalary
Status,,,,,
Churned,44.84,645.35,91108.54,4.93,101465.68
Retained,37.41,651.85,72745.30,5.03,99738.39


In [75]:
corr_cols = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts",
             "HasCrCard", "IsActiveMember", "EstimatedSalary", "Exited"]
corr = df[corr_cols].corr().round(2)
display(corr)
print("Correlation with Exited:")
print(corr["Exited"].drop("Exited").sort_values(ascending=False))

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
CreditScore,1.00,-0.00,0.00,0.01,0.01,-0.01,0.03,-0.00,-0.03
Age,-0.00,1.00,-0.01,0.03,-0.03,-0.01,0.09,-0.01,0.29
Tenure,0.00,-0.01,1.00,-0.01,0.01,0.02,-0.03,0.01,-0.01
Balance,0.01,0.03,-0.01,1.00,-0.30,-0.01,-0.01,0.01,0.12
NumOfProducts,0.01,-0.03,0.01,-0.30,1.00,0.00,0.01,0.01,-0.05
HasCrCard,-0.01,-0.01,0.02,-0.01,0.00,1.00,-0.01,-0.01,-0.01
IsActiveMember,0.03,0.09,-0.03,-0.01,0.01,-0.01,1.00,-0.01,-0.16
EstimatedSalary,-0.00,-0.01,0.01,0.01,0.01,-0.01,-0.01,1.00,0.01
Exited,-0.03,0.29,-0.01,0.12,-0.05,-0.01,-0.16,0.01,1.00


Correlation with Exited:
Age                0.29
Balance            0.12
EstimatedSalary    0.01
Tenure            -0.01
HasCrCard         -0.01
CreditScore       -0.03
NumOfProducts     -0.05
IsActiveMember    -0.16
Name: Exited, dtype: float64


### Identification and explanation of meaningful insights:
- **Geography:** churn is highest in Germany (32.44%) and lowest in Spain (16.67%), compared with the overall 20.37%.
- **Gender:** female churn is ____% against ____% for male customers, a gap of ____ percentage points.
- **Activity:** inactive members churn at ____% versus ____% for active members.
- **Number of products:** customers with 1 product churn at ____%, 2 products at ____%, and 3-4 products at ____%.
- **Age:** churn rises from ____% in the 18-30 group to a peak of ____% in the ____ group.
- **Churned vs retained:** churned customers are on average ____ years older (mean age ____ vs ____) and hold higher balances (____ vs ____).
- **Correlation:** the strongest correlation with `Exited` is `Age` (r = ____), followed by ____ (r = ____). Correlations between the other numerical attributes are weak.

**Note:** correlation does not prove causation. These results show association only and do not establish why customers left.

## 4. Data Visualization and Interpretation